In [1]:
!pip install git+https://github.com/leggedrobotics/rsl_rl.git
!pip install tensorboard h5py
!pip install tensordict

Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
  Cloning https://github.com/leggedrobotics/rsl_rl.git to /tmp/pip-req-build-f5vfkvdl
  Running command git clone --filter=blob:none --quiet https://github.com/leggedrobotics/rsl_rl.git /tmp/pip-req-build-f5vfkvdl
  Resolved https://github.com/leggedrobotics/rsl_rl.git to commit 3ac56acd3376f2952eb636a133f4b5aa30142552
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple


In [2]:
import shutil, os

paths = ["checkpoints/phase3", "logs/phase3", "checkpoints/bc_steer.pt"]

for path in paths:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
            print(f"Removed directory: {path}")
        else:
            os.remove(path)
            print(f"Removed file: {path}")
    else:
        print(f"Path not found, skipping: {path}")

Removed directory: checkpoints/phase3
Removed directory: logs/phase3
Removed file: checkpoints/bc_steer.pt


In [3]:
!pip install -q git+https://github.com/leggedrobotics/rsl_rl.git
!pip install -q tensorboard h5py tensordict

# =============================================================================
# PHASE-3  (BC + PPO)  — UPDATED VERSION v2
# =============================================================================
# What changed from the previous revision:
#
#  1) Curriculum thresholds lowered:
#       old level-0 gate required 45m, but policy plateaus around 42–43m
#       -> agent was stuck forever at 0% obstacles
#
#  2) Added stagnation-based forced curriculum promotion:
#       if progress does not improve for several evals, move up anyway
#
#  3) Reward scale reduced:
#       very large critic loss suggested value targets were too large
#       -> W_PROGRESS, milestones, lap bonus reduced moderately
#
#  4) Critic made less dominant:
#       lower value coefficient, lower gamma, lower horizon
#
#  5) Added explicit progress EMA tracking:
#       promotion uses smoothed progress, not one noisy eval
#
#  6) Best checkpoint now prioritizes:
#       success_rate > progress > score
#
# Expected behavior:
#   - curriculum should no longer remain frozen at level 0
#   - critic loss should be noticeably smaller
#   - progress may dip temporarily after level changes, which is acceptable
# =============================================================================

import os
import math
import inspect
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.distributions import Normal
from tensordict import TensorDict

from rsl_rl.algorithms import PPO
from rsl_rl.storage import RolloutStorage


# ===================== PATHS =====================
H5_PATH       = "datasets/phase2/combined_dataset.h5"
BC_CKPT_PATH  = "checkpoints/bc_steer.pt"
LOG_DIR       = "logs/phase3"
CKPT_DIR      = "checkpoints/phase3"

os.makedirs(os.path.dirname(BC_CKPT_PATH), exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)


# ===================== GLOBAL CONFIG =====================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED   = 0
np.random.seed(SEED)
torch.manual_seed(SEED)


# ===================== BC CONFIG =====================
BC_EPOCHS            = 150
BC_BATCH             = 512
BC_LR                = 5e-4
BC_WEIGHT_DECAY      = 1e-3
BC_PATIENCE          = 20
VAL_FRAC             = 0.2
DROP_COLLISION_STEPS = True
MIN_CLEARANCE_FILTER = 0.0


# ===================== PPO CONFIG =====================
NUM_ENVS    = 32
MAX_ITERS   = 6000
SAVE_FREQ   = 100
HORIZON     = 256

PPO_LR            = 8e-5
PPO_EPOCHS        = 4
PPO_MINI_BATCHES  = 8
PPO_CLIP          = 0.2
PPO_GAMMA         = 0.98
PPO_LAM           = 0.95
PPO_VALUE_COEF    = 0.25
PPO_ENTROPY_COEF  = 0.035
PPO_MAX_GRAD_NORM = 0.5
PPO_DESIRED_KL    = 0.01

EVAL_EPISODES_PER_SCENARIO = 4


# ===================== TRACK / ENV CONSTANTS =====================
OVAL_RADIUS  = 15.0
STRAIGHT_LEN = 40.0
TRACK_WIDTH  = 10.0
NUM_CONES    = 60
perimeter    = 2 * STRAIGHT_LEN + 2 * math.pi * OVAL_RADIUS

CONE_R      = 0.30
LANE_HALF   = TRACK_WIDTH / 2.0
CAR_HALF_W  = 1.25
LANE_SAFE   = float(LANE_HALF - (CAR_HALF_W + CONE_R + 0.25))
OFF_TRACK_THRESH = float(LANE_HALF - CAR_HALF_W - CONE_R)

OB_SIZE    = 1.5
MAX_STEER  = 0.55
BASE_SPEED = 8.0
MIN_SPEED  = 2.0
WHEELBASE  = 2.8

N_RAYS     = 41
FOV        = math.radians(170.0)
RAY_ANGLES = np.linspace(-FOV / 2, FOV / 2, N_RAYS).astype(np.float32)
LIDAR_MAX  = 30.0

NUM_OBS = 44
NUM_ACT = 1

MAX_EP_STEPS        = 420
COLLISION_THRESHOLD = 0.5
CRITICAL_THRESH     = 2.25


# ===================== REWARD WEIGHTS =====================
# Reduced overall scale to help critic stability.
W_PROGRESS  = 3.5
W_CTE       = 0.12
W_COLLISION = 8.0
W_SMOOTH    = 0.010
W_ACTION    = 0.0
W_STEP      = 0.0

R_LAP_BONUS      = 40.0
R_TIMEOUT        = 0.0
R_MILESTONE_25   = 4.0
R_MILESTONE_50   = 8.0
R_MILESTONE_75   = 12.0
R_MILESTONE_90   = 20.0


# ===================== CURRICULUM CONFIG =====================
# Lowered thresholds so the agent can actually leave warmup.
CURRICULUM_LEVELS = [
    {"name": "warmup_0",   "fraction": 0.00, "min_progress": 40.0, "hold_evals": 2},
    {"name": "warmup_10",  "fraction": 0.10, "min_progress": 36.0, "hold_evals": 2},
    {"name": "sparse_25",  "fraction": 0.25, "min_progress": 32.0, "hold_evals": 2},
    {"name": "medium_50",  "fraction": 0.50, "min_progress": 28.0, "hold_evals": 2},
    {"name": "dense_75",   "fraction": 0.75, "min_progress": 24.0, "hold_evals": 2},
    {"name": "full_100",   "fraction": 1.00, "min_progress": 22.0, "hold_evals": 999999},
]

REGRESSION_DROP_PROGRESS = 7.0
MIN_ITERS_BEFORE_LEVELUP = 200
MIN_ITERS_AFTER_LEVELUP  = 200

# New: if eval progress plateaus for this many evals, force one level up.
STAGNATION_EVALS_FOR_PROMOTION = 4
STAGNATION_MIN_DELTA = 0.75


# ===================== FILLED AFTER LOADING BC =====================
SIM_DT   = None
OBS_MEAN = None
OBS_STD  = None
ACT_MEAN = 0.0
ACT_STD  = 1.0


# ===================== HELPERS =====================
def wrap_pi(a):
    return (a + math.pi) % (2 * math.pi) - math.pi


def compute_yaw_rate(yaws: np.ndarray, dt: float) -> np.ndarray:
    T = len(yaws)
    yr = np.zeros((T,), dtype=np.float32)
    if T >= 2:
        for i in range(1, T):
            dy = wrap_pi(float(yaws[i]) - float(yaws[i - 1]))
            yr[i] = dy / float(dt)
    return yr


def get_oval_pos(dist, offset=0.0):
    d = dist % perimeter
    if d < STRAIGHT_LEN:
        return (OVAL_RADIUS + offset, -STRAIGHT_LEN / 2 + d)
    d -= STRAIGHT_LEN
    if d < math.pi * OVAL_RADIUS:
        ang = (d / (math.pi * OVAL_RADIUS)) * math.pi
        R = OVAL_RADIUS + offset
        return (R * math.cos(ang), STRAIGHT_LEN / 2 + R * math.sin(ang))
    d -= math.pi * OVAL_RADIUS
    if d < STRAIGHT_LEN:
        return (-OVAL_RADIUS - offset, STRAIGHT_LEN / 2 - d)
    d -= STRAIGHT_LEN
    ang = math.pi + (d / (math.pi * OVAL_RADIUS)) * math.pi
    R = OVAL_RADIUS + offset
    return (R * math.cos(ang), -STRAIGHT_LEN / 2 + R * math.sin(ang))


def get_tangent_normal(dist, eps=0.25):
    x1, y1 = get_oval_pos(dist, 0.0)
    x2, y2 = get_oval_pos(dist + eps, 0.0)
    tx, ty = x2 - x1, y2 - y1
    nrm = math.sqrt(tx * tx + ty * ty) + 1e-8
    tx, ty = tx / nrm, ty / nrm
    return tx, ty, -ty, tx


def project_to_centerline(px, py, s_guess, window=18.0, samples=81):
    best_s, best_d2 = s_guess, 1e30
    s0 = s_guess - window
    ds = (2 * window) / (samples - 1)
    for i in range(samples):
        s = s0 + i * ds
        cx, cy = get_oval_pos(s, 0.0)
        d2 = (px - cx) ** 2 + (py - cy) ** 2
        if d2 < best_d2:
            best_d2, best_s = d2, s
    best_s = best_s % perimeter
    cx, cy = get_oval_pos(best_s, 0.0)
    tx, ty, nx, ny = get_tangent_normal(best_s)
    cte = (px - cx) * nx + (py - cy) * ny
    return best_s, cx, cy, tx, ty, nx, ny, float(cte)


def build_cone_xy():
    cone_xy = []
    for i in range(NUM_CONES):
        d = (i / NUM_CONES) * perimeter
        for sign in (-1, 1):
            ex, ey = get_oval_pos(d, offset=sign * LANE_HALF)
            cone_xy.append((float(ex), float(ey), float(CONE_R)))
    return np.array(cone_xy, dtype=np.float32)


CONE_XY = build_cone_xy()


def get_obstacle_configs():
    return [
        {"name": "Baseline",
         "dists": [0.12, 0.22, 0.34, 0.48, 0.65, 0.85],
         "lats":  [+1.6, -1.6, +2.0, -2.0, +1.5, -1.5]},
        {"name": "Wider",
         "dists": [0.10, 0.25, 0.40, 0.55, 0.70, 0.90],
         "lats":  [+2.0, -2.0, +1.8, -1.8, +2.0, -2.0]},
        {"name": "Dense",
         "dists": [0.08, 0.18, 0.28, 0.40, 0.52, 0.64, 0.76, 0.88],
         "lats":  [+1.5, -1.5, +1.8, -1.8, +1.5, -1.5, +1.8, -1.8]},
        {"name": "Offset",
         "dists": [0.15, 0.30, 0.45, 0.60, 0.75],
         "lats":  [+1.9, -1.7, +1.6, -1.9, +1.7]},
    ]


def build_obs_boxes(config):
    boxes = []
    for d_frac, lat in zip(config["dists"], config["lats"]):
        d = d_frac * perimeter
        tx, ty, nx, ny = get_tangent_normal(d)
        cx, cy = get_oval_pos(d, 0.0)
        ox, oy = cx + lat * nx, cy + lat * ny
        boxes.append((float(ox), float(oy), OB_SIZE / 2, OB_SIZE / 2))
    return np.array(boxes, dtype=np.float32)


SCENARIO_CONFIGS = get_obstacle_configs()
ALL_OBS_BOXES    = [build_obs_boxes(c) for c in SCENARIO_CONFIGS]


def lidar_scan_fast(px, py, yaw, obs_boxes, max_range=30.0):
    if len(obs_boxes) == 0:
        return np.full(N_RAYS, max_range, dtype=np.float32)

    ga   = RAY_ANGLES + yaw
    dx   = np.sin(ga).astype(np.float32)
    dy   = np.cos(ga).astype(np.float32)
    P    = np.array([px, py], dtype=np.float32).reshape(1, 1, 2)
    D    = np.stack([dx, dy], axis=1).reshape(-1, 1, 2)
    B    = obs_boxes.reshape(1, -1, 4)
    Bmin = B[..., :2] - B[..., 2:]
    Bmax = B[..., :2] + B[..., 2:]

    invD    = 1.0 / (D + 1e-9)
    t0      = (Bmin - P) * invD
    t1      = (Bmax - P) * invD
    t_enter = np.max(np.minimum(t0, t1), axis=2)
    t_exit  = np.min(np.maximum(t0, t1), axis=2)
    hit     = (t_exit > t_enter) & (t_exit > 0)
    dists   = np.where(hit, np.maximum(0.0, t_enter), np.inf)
    return np.clip(np.min(dists, axis=1), 0.0, max_range).astype(np.float32)


def dist_point_to_aabb(px, py, cx, cy, hx, hy):
    return math.hypot(max(abs(px - cx) - hx, 0.0), max(abs(py - cy) - hy, 0.0))


def clearance_at_point(px, py, obs_boxes):
    best = 1e9
    for (cx, cy, hx, hy) in obs_boxes:
        best = min(best, dist_point_to_aabb(
            px, py, float(cx), float(cy), float(hx), float(hy)
        ))
    dx = CONE_XY[:, 0] - px
    dy = CONE_XY[:, 1] - py
    best = min(best, float(np.min(np.sqrt(dx * dx + dy * dy) - CONE_XY[:, 2])))
    return float(best)


def integrate_bicycle(px, py, yaw, v, delta, dt):
    yaw_rate = (v / WHEELBASE) * math.tan(delta)
    yaw  = wrap_pi(yaw + yaw_rate * dt)
    px  += v * math.sin(yaw) * dt
    py  += v * math.cos(yaw) * dt
    return px, py, yaw


# ===================== BC DATASET =====================
class BCDataset(Dataset):
    def __init__(self, h5_path, episode_names,
                 drop_collision_steps=True,
                 min_clearance=0.0,
                 norm=None):
        self.h5_path = h5_path
        self.episode_names = episode_names
        self.drop_collision_steps = bool(drop_collision_steps)
        self.min_clearance = float(min_clearance)

        with h5py.File(self.h5_path, "r") as f:
            self.dt = float(f.attrs["sim_dt"])

        self.index = []
        with h5py.File(self.h5_path, "r") as f:
            for epn in self.episode_names:
                g = f[epn]
                T = g["actions/steering"].shape[0]
                collisions = g["collisions"][...]
                clearances = g["metrics/clearances"][...]
                for t in range(T):
                    if self.drop_collision_steps and bool(collisions[t]):
                        continue
                    if self.min_clearance > 0.0 and float(clearances[t]) < self.min_clearance:
                        continue
                    self.index.append((epn, t))

        if len(self.index) == 0:
            raise RuntimeError("No samples after filtering.")

        if norm is None:
            self._fit_norm()
        else:
            self.obs_mean, self.obs_std, self.act_mean, self.act_std = norm

    def _fit_norm(self):
        sum_x = None
        sum_x2 = None
        sum_y = 0.0
        sum_y2 = 0.0
        n = 0

        with h5py.File(self.h5_path, "r") as f:
            epset = sorted(set(ep for ep, _ in self.index))
            for epn in epset:
                g = f[epn]
                lidar    = g["observations/lidar_ranges"][...].astype(np.float32)
                cte      = g["observations/cte"][...].astype(np.float32).reshape(-1, 1)
                speed    = g["velocities"][...].astype(np.float32).reshape(-1, 1)
                yaws     = g["yaws"][...].astype(np.float32)
                yaw_rate = compute_yaw_rate(yaws, self.dt).reshape(-1, 1)
                steer    = g["actions/steering"][...].astype(np.float32).reshape(-1, 1)

                obs_full = np.concatenate([lidar, cte, speed, yaw_rate], axis=1)
                ts  = [t for (e, t) in self.index if e == epn]
                obs = obs_full[ts]
                act = steer[ts]

                if sum_x is None:
                    sum_x  = obs.sum(axis=0)
                    sum_x2 = (obs ** 2).sum(axis=0)
                else:
                    sum_x  += obs.sum(axis=0)
                    sum_x2 += (obs ** 2).sum(axis=0)

                sum_y  += float(act.sum())
                sum_y2 += float((act ** 2).sum())
                n += obs.shape[0]

        mean_x = sum_x / max(n, 1)
        var_x  = sum_x2 / max(n, 1) - mean_x ** 2
        std_x  = np.sqrt(np.maximum(var_x, 1e-8))

        mean_y = sum_y / max(n, 1)
        var_y  = sum_y2 / max(n, 1) - mean_y ** 2
        std_y  = math.sqrt(max(var_y, 1e-8))

        self.obs_mean = mean_x.astype(np.float32)
        self.obs_std  = std_x.astype(np.float32)
        self.act_mean = np.array([mean_y], dtype=np.float32)
        self.act_std  = np.array([std_y],  dtype=np.float32)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        epn, t = self.index[i]
        with h5py.File(self.h5_path, "r") as f:
            g     = f[epn]
            lidar = g["observations/lidar_ranges"][t].astype(np.float32)
            cte   = np.array([g["observations/cte"][t]], dtype=np.float32)
            speed = np.array([g["velocities"][t]], dtype=np.float32)
            yaws  = g["yaws"]
            if t == 0:
                yr = 0.0
            else:
                yr = wrap_pi(float(yaws[t]) - float(yaws[t - 1])) / float(self.dt)
            yaw_rate = np.array([yr], dtype=np.float32)

            obs = np.concatenate([lidar, cte, speed, yaw_rate], axis=0)
            act = np.array([float(g["actions/steering"][t])], dtype=np.float32)

        obs_n = (obs - self.obs_mean) / self.obs_std
        act_n = (act - self.act_mean) / self.act_std
        return torch.from_numpy(obs_n), torch.from_numpy(act_n)


# ===================== NETWORKS =====================
class BCMLP(nn.Module):
    def __init__(self, in_dim=44, out_dim=1, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def normalized_action_bounds(act_mean, act_std, max_steer):
    lo = (-max_steer - act_mean) / act_std
    hi = ( max_steer - act_mean) / act_std
    return float(lo), float(hi)


class BoundedGaussianMLP(nn.Module):
    is_recurrent = False

    def __init__(self,
                 input_dim=NUM_OBS,
                 output_dim=NUM_ACT,
                 hidden_dims=(256, 256),
                 activation="relu",
                 init_log_std=-1.0,
                 norm_lo=-1.0,
                 norm_hi=1.0):
        super().__init__()
        act_fn = {"elu": nn.ELU, "relu": nn.ReLU, "tanh": nn.Tanh}[activation]
        layers, d = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(d, h), act_fn()]
            d = h
        layers.append(nn.Linear(d, output_dim))
        self.net     = nn.Sequential(*layers)
        self.log_std = nn.Parameter(torch.full((output_dim,), init_log_std))
        self.norm_lo = float(norm_lo)
        self.norm_hi = float(norm_hi)
        self._dist   = None
        self._mean   = None
        self._std    = None

    def _x(self, obs):
        return obs["policy"] if isinstance(obs, TensorDict) else obs

    def _bounded_mean(self, x):
        z = self.net(x)
        z = torch.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)
        return torch.clamp(z, self.norm_lo, self.norm_hi)

    def forward(self, obs, stochastic_output=False,
                masks=None, hidden_states=None, hidden_state=None):
        x    = self._x(obs)
        mean = self._bounded_mean(x)
        std  = torch.exp(torch.clamp(self.log_std, min=-4.0, max=1.0)).expand_as(mean)
        dist = Normal(mean, std)
        self._mean = mean
        self._std  = std
        self._dist = dist
        return dist.rsample() if stochastic_output else mean

    def get_output_log_prob(self, actions):
        return self._dist.log_prob(actions).sum(dim=-1, keepdim=True)

    def get_output_logprob(self, actions):
        return self.get_output_log_prob(actions)

    @property
    def output_distribution_params(self):
        return self._mean, self._std

    @property
    def output_entropy(self):
        return self._dist.entropy() if self._dist is not None else None

    def get_kl_divergence(self, old_params, new_params):
        old_mean, old_std = old_params
        new_mean, new_std = new_params
        old_dist = Normal(old_mean, old_std)
        new_dist = Normal(new_mean, new_std)
        return torch.distributions.kl_divergence(old_dist, new_dist).sum(dim=-1)

    def get_hidden_state(self):          return None
    def update_normalization(self, obs): pass
    def reset(self, dones=None):         pass


class ValueMLP(nn.Module):
    is_recurrent = False

    def __init__(self, input_dim=NUM_OBS, hidden_dims=(256, 256), activation="relu"):
        super().__init__()
        act_fn = {"elu": nn.ELU, "relu": nn.ReLU, "tanh": nn.Tanh}[activation]
        layers, d = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(d, h), act_fn()]
            d = h
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)

    def _x(self, obs):
        return obs["policy"] if isinstance(obs, TensorDict) else obs

    def forward(self, obs, masks=None, hidden_states=None, hidden_state=None):
        return self.net(self._x(obs))

    def get_hidden_state(self):          return None
    def update_normalization(self, obs): pass
    def reset(self, dones=None):         pass


# ===================== BC TRAINING =====================
def train_bc():
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    with h5py.File(H5_PATH, "r") as f:
        all_eps = sorted([k for k in f.keys() if k.startswith("episode_")])

    rng = np.random.RandomState(SEED)
    rng.shuffle(all_eps)
    n_val = max(1, int(round(VAL_FRAC * len(all_eps))))
    val_eps = all_eps[:n_val]
    train_eps = all_eps[n_val:]

    train_ds = BCDataset(
        H5_PATH, train_eps,
        drop_collision_steps=DROP_COLLISION_STEPS,
        min_clearance=MIN_CLEARANCE_FILTER,
        norm=None
    )
    norm = (train_ds.obs_mean, train_ds.obs_std, train_ds.act_mean, train_ds.act_std)
    val_ds = BCDataset(
        H5_PATH, val_eps,
        drop_collision_steps=DROP_COLLISION_STEPS,
        min_clearance=MIN_CLEARANCE_FILTER,
        norm=norm
    )

    train_dl = DataLoader(train_ds, batch_size=BC_BATCH, shuffle=True,
                          drop_last=True, num_workers=0)
    val_dl   = DataLoader(val_ds, batch_size=BC_BATCH, shuffle=False,
                          drop_last=False, num_workers=0)

    model   = BCMLP(44, 1).to(DEVICE)
    opt     = torch.optim.Adam(model.parameters(), lr=BC_LR, weight_decay=BC_WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    best_val = float("inf")
    no_improve = 0

    print("=" * 70)
    print("PHASE-3A  Behavior Cloning")
    print("=" * 70)
    print(f"Dataset : {H5_PATH}")
    print(f"Episodes: train={len(train_eps)} val={len(val_eps)}")
    print(f"Samples : train={len(train_ds)} val={len(val_ds)} | dt={train_ds.dt}")

    for ep in range(1, BC_EPOCHS + 1):
        model.train()
        tr = []
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tr.append(loss.item())

        model.eval()
        va = []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                va.append(loss_fn(model(xb), yb).item())

        tr_mse = float(np.mean(tr)) if tr else float("nan")
        va_mse = float(np.mean(va)) if va else float("nan")
        print(f"epoch {ep:03d} | train_mse={tr_mse:.6f} | val_mse={va_mse:.6f}")

        if va_mse < best_val:
            best_val = va_mse
            no_improve = 0
            ckpt = {
                "model_state": model.state_dict(),
                "obs_mean": train_ds.obs_mean,
                "obs_std": train_ds.obs_std,
                "act_mean": train_ds.act_mean,
                "act_std": train_ds.act_std,
                "dt": train_ds.dt,
                "obs_layout": "lidar(41), cte(1), speed(1), yaw_rate(1)",
                "target": "steering",
                "h5": H5_PATH,
                "filters": {
                    "drop_collision_steps": bool(DROP_COLLISION_STEPS),
                    "min_clearance": float(MIN_CLEARANCE_FILTER),
                }
            }
            torch.save(ckpt, BC_CKPT_PATH)
        else:
            no_improve += 1
            if no_improve >= BC_PATIENCE:
                print(f"  Early stop at epoch {ep} (no val improvement for {BC_PATIENCE} epochs)")
                break

    print(f"\nSaved BC checkpoint: {BC_CKPT_PATH}")
    print(f"Best BC val MSE: {best_val:.6f}")


# ===================== PPO ENV =====================
class AnalyticCarEnv:
    def __init__(self, num_envs=NUM_ENVS, device=DEVICE):
        self.num_envs   = num_envs
        self.device     = torch.device(device)

        self._base_configs = [ALL_OBS_BOXES[i % len(ALL_OBS_BOXES)]
                              for i in range(num_envs)]
        self.env_configs = [np.zeros((0, 4), dtype=np.float32)] * num_envs

        self.px          = np.zeros(num_envs, dtype=np.float32)
        self.py          = np.zeros(num_envs, dtype=np.float32)
        self.yaw         = np.zeros(num_envs, dtype=np.float32)
        self.speed       = np.full(num_envs, BASE_SPEED, dtype=np.float32)
        self.s_est       = np.zeros(num_envs, dtype=np.float32)
        self.prev_steer  = np.zeros(num_envs, dtype=np.float32)
        self.prev_yaw    = np.zeros(num_envs, dtype=np.float32)
        self.ep_steps    = np.zeros(num_envs, dtype=np.int32)
        self.ep_min_dist = np.full(num_envs, 1e9, dtype=np.float32)
        self.ep_dist     = np.zeros(num_envs, dtype=np.float32)

        self.milestone_awarded = np.zeros((num_envs, 4), dtype=bool)
        self._milestone_dists = [
            perimeter * 0.25,
            perimeter * 0.50,
            perimeter * 0.75,
            perimeter * 0.90,
        ]
        self._milestone_rewards = [
            R_MILESTONE_25,
            R_MILESTONE_50,
            R_MILESTONE_75,
            R_MILESTONE_90,
        ]

        self.obs_buf  = torch.zeros(num_envs, NUM_OBS, dtype=torch.float32, device=self.device)
        self.rew_buf  = torch.zeros(num_envs, dtype=torch.float32, device=self.device)
        self.done_buf = torch.zeros(num_envs, dtype=torch.bool, device=self.device)

        self.success_log   = []
        self.min_dist_log  = []
        self.collision_log = []
        self.progress_log  = []

    def set_curriculum(self, fraction: float):
        for i in range(self.num_envs):
            boxes = self._base_configs[i]
            n = max(1, int(math.ceil(len(boxes) * fraction))) if fraction > 0 else 0
            self.env_configs[i] = boxes[:n] if n > 0 else np.zeros((0, 4), dtype=np.float32)

    def _reset_env(self, i):
        s_start = np.random.uniform(5.0, 13.0)
        cx0, cy0 = get_oval_pos(s_start, 0.0)
        tx, ty, nx, ny = get_tangent_normal(s_start)
        lat_noise = np.random.uniform(-0.08, 0.08)

        self.px[i]  = float(cx0) + lat_noise * nx
        self.py[i]  = float(cy0) + lat_noise * ny
        self.yaw[i] = float(math.atan2(tx, ty)) + np.random.uniform(-0.025, 0.025)

        self.s_est[i]       = s_start
        self.speed[i]       = BASE_SPEED
        self.prev_steer[i]  = 0.0
        self.prev_yaw[i]    = self.yaw[i]
        self.ep_steps[i]    = 0
        self.ep_min_dist[i] = 1e9
        self.ep_dist[i]     = 0.0
        self.milestone_awarded[i] = False

    def reset(self):
        for i in range(self.num_envs):
            self._reset_env(i)
        return self._compute_obs()

    def _compute_obs(self):
        obs_np = np.zeros((self.num_envs, NUM_OBS), dtype=np.float32)
        for i in range(self.num_envs):
            s_est, _, _, _, _, _, _, cte = project_to_centerline(
                self.px[i], self.py[i], self.s_est[i], window=18.0, samples=81
            )
            self.s_est[i] = s_est

            ranges   = lidar_scan_fast(self.px[i], self.py[i], self.yaw[i],
                                       self.env_configs[i], LIDAR_MAX)
            yaw_rate = (self.speed[i] / WHEELBASE) * math.tan(float(self.prev_steer[i]))

            obs_np[i, :N_RAYS]    = ranges
            obs_np[i, N_RAYS]     = cte
            obs_np[i, N_RAYS + 1] = self.speed[i]
            obs_np[i, N_RAYS + 2] = yaw_rate

        obs_t = torch.from_numpy(obs_np).to(self.device)
        obs_t = (obs_t - OBS_MEAN) / OBS_STD
        self.obs_buf.copy_(obs_t)
        return self.obs_buf

    def step(self, actions: torch.Tensor):
        acts_np  = actions.detach().cpu().numpy().reshape(-1)
        rews_np  = np.zeros(self.num_envs, dtype=np.float32)
        dones_np = np.zeros(self.num_envs, dtype=bool)

        for i in range(self.num_envs):
            prev_steer_old = float(self.prev_steer[i])

            raw_act = float(acts_np[i]) * ACT_STD + ACT_MEAN
            steer   = float(np.clip(raw_act, -MAX_STEER, MAX_STEER))

            s_old     = self.s_est[i]
            clearance = clearance_at_point(self.px[i], self.py[i], self.env_configs[i])

            speed = MIN_SPEED if clearance < CRITICAL_THRESH else BASE_SPEED
            self.speed[i] = speed

            px_new, py_new, yaw_new = integrate_bicycle(
                self.px[i], self.py[i], self.yaw[i], speed, steer, SIM_DT
            )

            s_new, _, _, _, _, _, _, cte = project_to_centerline(
                px_new, py_new, s_old, window=18.0, samples=81
            )

            self.px[i], self.py[i], self.yaw[i] = px_new, py_new, yaw_new
            self.s_est[i]    = s_new
            self.prev_yaw[i] = yaw_new
            self.ep_steps[i] += 1

            clearance = clearance_at_point(self.px[i], self.py[i], self.env_configs[i])
            self.ep_min_dist[i] = min(self.ep_min_dist[i], clearance)

            off_track = abs(cte) > OFF_TRACK_THRESH

            ds = s_new - s_old
            if ds > perimeter / 2:
                ds -= perimeter
            elif ds < -perimeter / 2:
                ds += perimeter
            ds_fwd = max(0.0, float(ds))
            self.ep_dist[i] += ds_fwd

            collision = clearance < COLLISION_THRESHOLD
            lap_done  = self.ep_dist[i] >= perimeter * 0.95
            timeout   = self.ep_steps[i] >= MAX_EP_STEPS
            done      = collision or timeout or lap_done or off_track

            r_prog    = W_PROGRESS * ds_fwd
            r_smooth  = -W_SMOOTH * (steer - prev_steer_old) ** 2
            r_cte     = -W_CTE * (cte / LANE_SAFE) ** 2
            r_action  = -W_ACTION * (steer / MAX_STEER) ** 2
            r_col     = -W_COLLISION if (collision or off_track) else 0.0
            r_step    = -W_STEP
            r_lap     = +R_LAP_BONUS if lap_done else 0.0
            r_timeout = -R_TIMEOUT if (timeout and not lap_done and not collision) else 0.0

            r_milestone = 0.0
            for m_idx, (m_dist, m_rew) in enumerate(zip(self._milestone_dists, self._milestone_rewards)):
                if (not self.milestone_awarded[i, m_idx]) and (self.ep_dist[i] >= m_dist):
                    r_milestone += m_rew
                    self.milestone_awarded[i, m_idx] = True

            rews_np[i]  = r_prog + r_cte + r_smooth + r_action + r_col + r_step + r_lap + r_timeout + r_milestone
            dones_np[i] = done
            self.prev_steer[i] = steer

            if done:
                self.success_log.append(bool(lap_done))
                self.min_dist_log.append(float(self.ep_min_dist[i]))
                self.collision_log.append(bool(collision or off_track))
                self.progress_log.append(float(self.ep_dist[i]))
                self.ep_min_dist[i] = 1e9
                self.ep_dist[i]     = 0.0
                self._reset_env(i)

        self.rew_buf.copy_(torch.from_numpy(rews_np))
        self.done_buf.copy_(torch.from_numpy(dones_np))
        return self._compute_obs(), self.rew_buf, self.done_buf

    def print_stats(self):
        n = len(self.success_log)
        if n == 0:
            print("No completed episodes.")
            return
        print("=" * 60)
        print(f"PPO training stats over {n} completed episodes")
        print("=" * 60)
        print(f"  Lap completion rate: {100 * np.mean(self.success_log):.1f}%")
        print(f"  Collision rate     : {100 * np.mean(self.collision_log):.1f}%")
        print(f"  Mean min clearance : {np.mean(self.min_dist_log):.3f} m")
        print(f"  Mean progress      : {np.mean(self.progress_log):.3f} m")
        print("=" * 60)


# ===================== EVAL =====================
def eval_policy_deterministic(actor, obstacle_fraction=1.0, num_eps_per_scenario=EVAL_EPISODES_PER_SCENARIO):
    actor.eval()
    rows = []

    for sc_cfg, full_obs_boxes in zip(SCENARIO_CONFIGS, ALL_OBS_BOXES):
        n = max(1, int(math.ceil(len(full_obs_boxes) * obstacle_fraction))) if obstacle_fraction > 0 else 0
        obs_boxes = full_obs_boxes[:n] if n > 0 else np.zeros((0, 4), dtype=np.float32)

        for _ in range(num_eps_per_scenario):
            s_start = np.random.uniform(5.0, 13.0)
            cx0, cy0 = get_oval_pos(s_start, 0.0)
            tx, ty, nx, ny = get_tangent_normal(s_start)
            lat_noise = np.random.uniform(-0.2, 0.2)

            px    = float(cx0) + lat_noise * nx
            py    = float(cy0) + lat_noise * ny
            yaw   = float(math.atan2(tx, ty)) + np.random.uniform(-0.025, 0.025)
            s_est = s_start

            speed      = BASE_SPEED
            prev_steer = 0.0
            ep_dist    = 0.0
            min_clear  = 1e9
            collision  = False
            lap_done   = False
            timeout    = False
            off_track  = False

            for step in range(MAX_EP_STEPS):
                s_est, _, _, _, _, _, _, cte = project_to_centerline(
                    px, py, s_est, window=18.0, samples=81
                )
                ranges   = lidar_scan_fast(px, py, yaw, obs_boxes, LIDAR_MAX)
                yaw_rate = (speed / WHEELBASE) * math.tan(prev_steer)

                raw_obs = np.zeros((NUM_OBS,), dtype=np.float32)
                raw_obs[:N_RAYS]    = ranges
                raw_obs[N_RAYS]     = cte
                raw_obs[N_RAYS + 1] = speed
                raw_obs[N_RAYS + 2] = yaw_rate

                obs_t = torch.from_numpy(raw_obs).to(DEVICE).unsqueeze(0)
                obs_t = (obs_t - OBS_MEAN) / OBS_STD

                with torch.no_grad():
                    act_n   = actor(obs_t, stochastic_output=False)
                    steer_u = float((act_n * ACT_STD + ACT_MEAN)[0, 0].item())
                    steer   = float(np.clip(steer_u, -MAX_STEER, MAX_STEER))

                s_old     = s_est
                clearance = clearance_at_point(px, py, obs_boxes)
                speed     = MIN_SPEED if clearance < CRITICAL_THRESH else BASE_SPEED

                px_new, py_new, yaw_new = integrate_bicycle(px, py, yaw, speed, steer, SIM_DT)
                s_new, _, _, _, _, _, _, cte = project_to_centerline(
                    px_new, py_new, s_old, window=18.0, samples=81
                )

                px, py, yaw = px_new, py_new, yaw_new
                prev_steer  = steer
                s_est       = s_new

                clearance = clearance_at_point(px, py, obs_boxes)
                min_clear = min(min_clear, clearance)
                off_track = abs(cte) > OFF_TRACK_THRESH

                ds = s_new - s_old
                if ds > perimeter / 2:
                    ds -= perimeter
                elif ds < -perimeter / 2:
                    ds += perimeter
                ep_dist += max(0.0, float(ds))

                collision = clearance < COLLISION_THRESHOLD
                lap_done  = ep_dist >= perimeter * 0.95
                timeout   = not collision and not off_track and not lap_done and (step + 1 >= MAX_EP_STEPS)

                if collision or off_track or lap_done or timeout:
                    break

            rows.append({
                "scenario": sc_cfg["name"],
                "collision": int(collision or off_track),
                "timeout": int(timeout),
                "lap_completed": int(lap_done),
                "success": int(lap_done),
                "progress_m": float(ep_dist),
                "min_clearance_m": float(min_clear),
            })

    df = pd.DataFrame(rows)
    overall = {
        "success_rate":    float(df["success"].mean()),
        "collision_rate":  float(df["collision"].mean()),
        "timeout_rate":    float(df["timeout"].mean()),
        "mean_progress_m": float(df["progress_m"].mean()),
        "mean_min_clearance_m": float(df["min_clearance_m"].mean()),
        "score": float(
            400.0 * df["success"].mean()
            + 8.0  * (df["progress_m"] >= perimeter * 0.25).mean()
            + 12.0 * (df["progress_m"] >= perimeter * 0.50).mean()
            + 18.0 * (df["progress_m"] >= perimeter * 0.75).mean()
            + 24.0 * (df["progress_m"] >= perimeter * 0.90).mean()
            + 12.0 * (df["progress_m"].clip(lower=10.0).mean() / perimeter)
            + 3.0  * np.clip(df["min_clearance_m"].mean(), 0.0, 2.0)
            - 35.0 * df["collision"].mean()
        ),
    }
    return df, overall


# ===================== PPO INFRA =====================
def tot_d(obs_tensor):
    return TensorDict({"policy": obs_tensor},
                      batch_size=obs_tensor.shape[0],
                      device=obs_tensor.device)


def has_nan_weights(model):
    return any(torch.isnan(p).any() for p in model.parameters())


def reload_checkpoint(path, actor, critic, ppo):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    actor.load_state_dict(ckpt["actor"])
    critic.load_state_dict(ckpt["critic"])
    if "optimizer" in ckpt:
        ppo.optimizer.load_state_dict(ckpt["optimizer"])
    for g in ppo.optimizer.param_groups:
        g["lr"] = PPO_LR
    ppo.learning_rate = PPO_LR
    print(f"Reloaded checkpoint after NaN/regression: {path}")


def build_storage_and_ppo(actor, critic):
    rs_params  = inspect.signature(RolloutStorage.__init__).parameters
    ppo_params = inspect.signature(PPO.__init__).parameters

    obs_proto = TensorDict(
        {"policy": torch.zeros(NUM_ENVS, NUM_OBS, device=DEVICE)},
        batch_size=[NUM_ENVS], device=DEVICE
    )

    if "obs" in rs_params:
        storage = RolloutStorage(
            training_type="rl",
            num_envs=NUM_ENVS,
            num_transitions_per_env=HORIZON,
            obs=obs_proto,
            actions_shape=(NUM_ACT,),
            device=DEVICE
        )
    elif "obs_shape" in rs_params:
        storage = RolloutStorage(
            training_type="rl",
            num_envs=NUM_ENVS,
            num_transitions_per_env=HORIZON,
            obs_shape={"policy": (NUM_OBS,)},
            actions_shape=(NUM_ACT,),
            device=DEVICE
        )
    else:
        storage = RolloutStorage(
            num_envs=NUM_ENVS,
            num_transitions_per_env=HORIZON,
            num_obs=NUM_OBS,
            num_actions=NUM_ACT,
            device=DEVICE
        )

    ppo_kwargs = dict(
        actor=actor,
        critic=critic,
        storage=storage,
        num_learning_epochs=PPO_EPOCHS,
        num_mini_batches=PPO_MINI_BATCHES,
        clip_param=PPO_CLIP,
        gamma=PPO_GAMMA,
        lam=PPO_LAM,
        value_loss_coef=PPO_VALUE_COEF,
        entropy_coef=PPO_ENTROPY_COEF,
        learning_rate=PPO_LR,
        max_grad_norm=PPO_MAX_GRAD_NORM,
        use_clipped_value_loss=True,
        schedule="adaptive",
        desired_kl=PPO_DESIRED_KL,
        device=DEVICE,
    )

    if "optimizer" in ppo_params:
        ppo_kwargs["optimizer"] = "adam"

    ppo = PPO(**ppo_kwargs)

    pep = inspect.signature(ppo.process_env_step).parameters
    if "infos" in pep:
        ppo._pep_mode = "infos"
    elif "extras" in pep:
        ppo._pep_mode = "extras"
    else:
        ppo._pep_mode = "none"

    return storage, ppo


def curriculum_status(level_idx):
    lvl = CURRICULUM_LEVELS[level_idx]
    return f"{lvl['name']} ({int(100*lvl['fraction'])}%)"


# ===================== MAIN PPO =====================
def train_ppo():
    global OBS_MEAN, OBS_STD, ACT_MEAN, ACT_STD, SIM_DT

    print("=" * 70)
    print("PHASE-3B  PPO fine-tuning")
    print("=" * 70)

    assert os.path.exists(BC_CKPT_PATH), f"BC checkpoint not found: {BC_CKPT_PATH}"
    bc_ckpt = torch.load(BC_CKPT_PATH, map_location=DEVICE, weights_only=False)

    OBS_MEAN = torch.from_numpy(bc_ckpt["obs_mean"].astype(np.float32)).to(DEVICE)
    OBS_STD  = torch.from_numpy(bc_ckpt["obs_std"].astype(np.float32)).to(DEVICE)
    ACT_MEAN = float(bc_ckpt["act_mean"][0])
    ACT_STD  = float(bc_ckpt["act_std"][0])
    SIM_DT   = float(bc_ckpt["dt"])

    norm_lo, norm_hi = normalized_action_bounds(ACT_MEAN, ACT_STD, MAX_STEER)

    print(f"Using dataset dt from BC checkpoint: {SIM_DT}")
    print(f"Normalized action bounds: [{norm_lo:.4f}, {norm_hi:.4f}]")

    actor = BoundedGaussianMLP(
        input_dim=NUM_OBS,
        output_dim=NUM_ACT,
        hidden_dims=(256, 256),
        activation="relu",
        init_log_std=-1.0,
        norm_lo=norm_lo,
        norm_hi=norm_hi,
    ).to(DEVICE)

    critic = ValueMLP(NUM_OBS, (256, 256), "relu").to(DEVICE)

    with torch.no_grad():
        actor.net[-1].weight.zero_()
        zero_steer_bias = float(np.clip(-ACT_MEAN / ACT_STD, norm_lo, norm_hi))
        actor.net[-1].bias.fill_(zero_steer_bias)

    print(f"PPO actor: zero-weight, bias={zero_steer_bias:.4f} -> ~0 rad steer on straights")
    print("Transferred BC tensors to PPO actor: 0/6 (intentional)")

    storage, ppo = build_storage_and_ppo(actor, critic)
    env = AnalyticCarEnv(NUM_ENVS, DEVICE)

    curriculum_level = 0
    level_enter_iter = 0
    level_hold_counter = 0
    last_good_ckpt = None

    best_eval_score = -1e18
    best_eval_progress = -1e18
    best_success_rate = -1e18

    progress_ema = None
    stagnation_count = 0
    prev_progress_ema = None

    env.set_curriculum(CURRICULUM_LEVELS[curriculum_level]["fraction"])
    obs = env.reset()
    obs_td = tot_d(obs)

    writer = SummaryWriter(LOG_DIR)

    print(
        f"{'iter':>6s} {'rew':>9s} {'surr':>10s} {'value':>10s} "
        f"{'done':>8s} {'mind':>8s} {'progress':>9s} {'lap%':>6s} "
        f"{'obs%':>5s} {'lvl':>10s} {'eval':>8s}"
    )

    for it in range(MAX_ITERS):
        current_frac = CURRICULUM_LEVELS[curriculum_level]["fraction"]
        env.set_curriculum(current_frac)

        with torch.no_grad():
            for _ in range(HORIZON):
                actions = ppo.act(obs_td)
                obs_next, rewards, dones = env.step(actions)
                obs_next_td = tot_d(obs_next)

                if ppo._pep_mode == "infos":
                    ppo.process_env_step(obs_next_td, rewards, dones, infos={"time_outs": dones})
                elif ppo._pep_mode == "extras":
                    ppo.process_env_step(obs_next_td, rewards, dones, {"time_outs": dones})
                else:
                    ppo.process_env_step(obs_next_td, rewards, dones)

                obs_td = obs_next_td

            ppo.compute_returns(obs_td)

        if has_nan_weights(actor) or has_nan_weights(critic):
            if last_good_ckpt is not None:
                reload_checkpoint(last_good_ckpt, actor, critic, ppo)
            else:
                with torch.no_grad():
                    actor.net[-1].weight.zero_()
                    actor.net[-1].bias.fill_(float(np.clip(-ACT_MEAN / ACT_STD, norm_lo, norm_hi)))
            obs = env.reset()
            obs_td = tot_d(obs)
            continue

        loss_dict = ppo.update()
        surr = float(loss_dict.get("surrogate", loss_dict.get("policy_loss", 0.0)))
        val  = float(loss_dict.get("value", loss_dict.get("value_loss", 0.0)))
        entr = float(loss_dict.get("entropy", loss_dict.get("entropy_loss", 0.0)))

        mean_rew  = float(env.rew_buf.mean())
        done_rate = float(env.done_buf.float().mean())
        mean_mind = float(np.mean(env.min_dist_log[-NUM_ENVS:])) if len(env.min_dist_log) >= NUM_ENVS else 0.0
        mean_prog = float(np.mean(env.progress_log[-NUM_ENVS:])) if len(env.progress_log) >= NUM_ENVS else 0.0
        lap_rate  = float(np.mean(env.success_log[-NUM_ENVS:])) if len(env.success_log) >= NUM_ENVS else 0.0

        writer.add_scalar("train/reward", mean_rew, it)
        writer.add_scalar("train/surrogate_loss", surr, it)
        writer.add_scalar("train/value_loss", val, it)
        writer.add_scalar("train/entropy", entr, it)
        writer.add_scalar("train/done_rate", done_rate, it)
        writer.add_scalar("train/mean_min_dist", mean_mind, it)
        writer.add_scalar("train/mean_progress", mean_prog, it)
        writer.add_scalar("train/lap_rate_recent", lap_rate, it)
        writer.add_scalar("train/lr", ppo.learning_rate, it)
        writer.add_scalar("curriculum/level", curriculum_level, it)
        writer.add_scalar("curriculum/fraction", current_frac, it)

        eval_score = float("nan")

        if it % SAVE_FREQ == 0 and it > 0:
            eval_df, eval_overall = eval_policy_deterministic(
                actor,
                obstacle_fraction=current_frac,
                num_eps_per_scenario=EVAL_EPISODES_PER_SCENARIO,
            )
            eval_score    = eval_overall["score"]
            eval_progress = eval_overall["mean_progress_m"]
            eval_success  = eval_overall["success_rate"]

            if progress_ema is None:
                progress_ema = eval_progress
            else:
                progress_ema = 0.7 * progress_ema + 0.3 * eval_progress

            if prev_progress_ema is not None and abs(progress_ema - prev_progress_ema) < STAGNATION_MIN_DELTA:
                stagnation_count += 1
            else:
                stagnation_count = 0
            prev_progress_ema = progress_ema

            ckpt_payload = {
                "iteration": it,
                "actor": actor.state_dict(),
                "critic": critic.state_dict(),
                "optimizer": ppo.optimizer.state_dict(),
                "lr": ppo.learning_rate,
                "obs_mean": bc_ckpt["obs_mean"],
                "obs_std": bc_ckpt["obs_std"],
                "act_mean": bc_ckpt["act_mean"],
                "act_std": bc_ckpt["act_std"],
                "dt": SIM_DT,
                "phase3_meta": {
                    "base_speed": BASE_SPEED,
                    "min_speed": MIN_SPEED,
                    "max_steer": MAX_STEER,
                    "collision_threshold": COLLISION_THRESHOLD,
                    "critical_thresh": CRITICAL_THRESH,
                    "max_ep_steps": MAX_EP_STEPS,
                    "lane_safe": LANE_SAFE,
                    "sim_dt": SIM_DT,
                    "actor_type": "bounded_gaussian_mlp",
                    "norm_lo": norm_lo,
                    "norm_hi": norm_hi,
                    "curriculum_level": curriculum_level,
                    "curriculum_name": CURRICULUM_LEVELS[curriculum_level]["name"],
                    "curriculum_fraction": current_frac,
                    "reward": {
                        "W_PROGRESS": W_PROGRESS,
                        "W_CTE": W_CTE,
                        "W_COLLISION": W_COLLISION,
                        "W_SMOOTH": W_SMOOTH,
                        "W_ACTION": W_ACTION,
                        "W_STEP": W_STEP,
                        "R_LAP_BONUS": R_LAP_BONUS,
                        "R_TIMEOUT": R_TIMEOUT,
                        "R_MILESTONE_25": R_MILESTONE_25,
                        "R_MILESTONE_50": R_MILESTONE_50,
                        "R_MILESTONE_75": R_MILESTONE_75,
                        "R_MILESTONE_90": R_MILESTONE_90,
                    },
                },
                "eval_overall": eval_overall,
            }

            ckpt_path = os.path.join(CKPT_DIR, f"ckpt{it:06d}.pth")
            torch.save(ckpt_payload, ckpt_path)
            torch.save(ckpt_payload, os.path.join(CKPT_DIR, "latest.pth"))

            if eval_progress >= best_eval_progress - 1.5:
                last_good_ckpt = ckpt_path

            is_new_best = False
            if eval_success > best_success_rate:
                is_new_best = True
            elif eval_success == best_success_rate and eval_progress > best_eval_progress + 0.25:
                is_new_best = True
            elif eval_success == best_success_rate and eval_progress >= best_eval_progress - 0.25 and eval_score > best_eval_score:
                is_new_best = True

            if is_new_best:
                best_success_rate = eval_success
                best_eval_progress = max(best_eval_progress, eval_progress)
                best_eval_score = eval_score
                torch.save(ckpt_payload, os.path.join(CKPT_DIR, "best.pth"))
                print(
                    f"  best checkpoint updated at iter {it} | "
                    f"eval_score={eval_score:.3f} | eval_progress={eval_progress:.2f} | eval_success={eval_success:.3f}"
                )

            # performance-gated curriculum
            lvl_cfg = CURRICULUM_LEVELS[curriculum_level]
            enough_time_on_level = (it - level_enter_iter) >= MIN_ITERS_BEFORE_LEVELUP

            if progress_ema >= lvl_cfg["min_progress"] and enough_time_on_level:
                level_hold_counter += 1
            else:
                level_hold_counter = 0

            should_force_promote = (
                stagnation_count >= STAGNATION_EVALS_FOR_PROMOTION and
                curriculum_level < len(CURRICULUM_LEVELS) - 1 and
                enough_time_on_level
            )

            if ((level_hold_counter >= lvl_cfg["hold_evals"]) or should_force_promote):
                if curriculum_level < len(CURRICULUM_LEVELS) - 1:
                    old_level = curriculum_level
                    curriculum_level += 1
                    level_enter_iter = it
                    level_hold_counter = 0
                    stagnation_count = 0
                    print(
                        f"  curriculum advanced at iter {it}: "
                        f"{curriculum_status(old_level)} -> {curriculum_status(curriculum_level)}"
                    )

            # regression recovery
            if curriculum_level > 0 and (it - level_enter_iter) >= MIN_ITERS_AFTER_LEVELUP:
                cur_target = CURRICULUM_LEVELS[curriculum_level]["min_progress"]
                if progress_ema < (cur_target - REGRESSION_DROP_PROGRESS):
                    prev_level = curriculum_level
                    curriculum_level -= 1
                    level_enter_iter = it
                    level_hold_counter = 0
                    stagnation_count = 0
                    print(
                        f"  curriculum regression at iter {it}: "
                        f"{curriculum_status(prev_level)} -> {curriculum_status(curriculum_level)}"
                    )
                    if last_good_ckpt is not None:
                        reload_checkpoint(last_good_ckpt, actor, critic, ppo)
                    env.set_curriculum(CURRICULUM_LEVELS[curriculum_level]["fraction"])
                    obs = env.reset()
                    obs_td = tot_d(obs)

            writer.add_scalar("eval/score", eval_score, it)
            writer.add_scalar("eval/mean_progress_m", eval_progress, it)
            writer.add_scalar("eval/progress_ema_m", progress_ema, it)
            writer.add_scalar("eval/success_rate", eval_success, it)
            writer.add_scalar("eval/collision_rate", eval_overall["collision_rate"], it)
            writer.add_scalar("eval/timeout_rate", eval_overall["timeout_rate"], it)

        if it % 10 == 0:
            print(
                f"{it:6d} {mean_rew:9.3f} {surr:10.4f} {val:10.4f} "
                f"{done_rate:8.3f} {mean_mind:8.3f} {mean_prog:8.1f}m "
                f"{100*lap_rate:5.1f}% {100*current_frac:4.0f}% "
                f"{curriculum_level:10d} {eval_score:8.3f}"
            )

    writer.close()
    env.print_stats()

    final_path = os.path.join(CKPT_DIR, "final.pth")
    torch.save({
        "iteration": MAX_ITERS,
        "actor": actor.state_dict(),
        "critic": critic.state_dict(),
        "obs_mean": bc_ckpt["obs_mean"],
        "obs_std": bc_ckpt["obs_std"],
        "act_mean": bc_ckpt["act_mean"],
        "act_std": bc_ckpt["act_std"],
        "dt": SIM_DT,
    }, final_path)

    print(f"\nSaved final checkpoint: {final_path}")
    best_path = os.path.join(CKPT_DIR, "best.pth")
    if os.path.exists(best_path):
        print(f"Best PPO checkpoint: {best_path}")


# ===================== MAIN =====================
if __name__ == "__main__":
    train_bc()
    train_ppo()

PHASE-3A  Behavior Cloning
Dataset : datasets/phase2/combined_dataset.h5
Episodes: train=32 val=8
Samples : train=9600 val=2400 | dt=0.05
epoch 001 | train_mse=0.669846 | val_mse=0.535522
epoch 002 | train_mse=0.384319 | val_mse=0.432528
epoch 003 | train_mse=0.317035 | val_mse=0.405729
epoch 004 | train_mse=0.289173 | val_mse=0.404288
epoch 005 | train_mse=0.271345 | val_mse=0.373966
epoch 006 | train_mse=0.257689 | val_mse=0.372633
epoch 007 | train_mse=0.245594 | val_mse=0.359353
epoch 008 | train_mse=0.235368 | val_mse=0.378887
epoch 009 | train_mse=0.227429 | val_mse=0.347343
epoch 010 | train_mse=0.216155 | val_mse=0.353434
epoch 011 | train_mse=0.210106 | val_mse=0.352867
epoch 012 | train_mse=0.208075 | val_mse=0.353395
epoch 013 | train_mse=0.201309 | val_mse=0.333704
epoch 014 | train_mse=0.195126 | val_mse=0.339021
epoch 015 | train_mse=0.192133 | val_mse=0.341982
epoch 016 | train_mse=0.186702 | val_mse=0.351201
epoch 017 | train_mse=0.186546 | val_mse=0.349303
epoch 018 | 